In [0]:
%pip install folium requests openaq xgboost --quiet

In [0]:
# ── Standard library ──────────────────────────────────────────────
import os
import json
import time
import datetime
from datetime import timedelta

# ── Numerical & data ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── API & HTTP ────────────────────────────────────────────────────
import requests
import openaq

# ── Machine learning ──────────────────────────────────────────────
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── MLflow ────────────────────────────────────────────────────────
import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature

# ── Visualisation ─────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium

# ── Databricks / Spark ────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DateType

# ── Display settings ──────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

In [0]:
# ── Global path variables ─────────────────────────────────────────
AQI_PATH        = "/Volumes/workspace/bronze/raw_data/aqi/city_day.csv"
HEALTH_PATH     = "/Volumes/workspace/bronze/raw_data/health/geocode_health_centre.csv"
# NFHS_PATH       = "/Volumes/workspace/bronze/raw_data/nfhs"
CENSUS_PATH     = "/Volumes/workspace/bronze/raw_data/census/populaion_testing - Sheet1.csv"

# ── Load all files ────────────────────────────────────────────────
aqi_df     = spark.read.csv(AQI_PATH,    header=True, inferSchema=True)
health_df  = spark.read.csv(HEALTH_PATH, header=True, inferSchema=True)
# nfhs_df    = spark.read.csv(NFHS_PATH,   header=True, inferSchema=True)
census_df  = spark.read.csv(CENSUS_PATH, header=True, inferSchema=True)

aqi_df = aqi_df.withColumnRenamed("PM2.5", "PM2_5")
# ── Inspect each file ─────────────────────────────────────────────
for name, df in [("AQI", aqi_df),
                 ("Health", health_df), 
                 ("Census", census_df)]:
    print(f"\n{'='*50}")
    print(f"{name} — shape: ({df.count()}, {len(df.columns)})")
    print(f"{'='*50}")
    df.printSchema()
    display(df.limit(5))

# ── Basic info ────────────────────────────────────────────────────
print(f"Total rows     : {aqi_df.count()}")
print(f"Total columns  : {len(aqi_df.columns)}")
print(f"Columns        : {aqi_df.columns}")

# # ── Null count per column ─────────────────────────────────────────
# from pyspark.sql.functions import col, count, when

# null_counts = aqi_df.select([
#     count(when(col(f"`{c}`").isNull(), c)).alias(c) 
#     for c in aqi_df.columns
# ])
# display(null_counts)

# ── Sample data ───────────────────────────────────────────────────
# display(aqi_df.limit(25))

In [0]:
from pyspark.sql.functions import col, mean, when, count, trim
from pyspark.sql.functions import year, month, dayofweek
import pyspark.sql.functions as F

# ════════════════════════════════════════════════════════
# 1. CLEAN AQI DATA
# ════════════════════════════════════════════════════════

pollutant_cols = ["PM2_5","PM10","NO","NO2","NOx","NH3","CO","SO2","O3","Benzene","Toluene","Xylene"]

# Drop rows where all pollutants are null
aqi_clean = aqi_df.dropna(how="all", subset=pollutant_cols)

# Fill null pollutant values with city-level mean
for c in pollutant_cols:
    safe_c     = f"`{c}`"
    alias_c    = c.replace(".", "_")
    city_means = aqi_clean.groupBy("City").agg(mean(col(safe_c)).alias(f"{alias_c}_mean"))
    aqi_clean  = aqi_clean.join(city_means, on="City", how="left")
    aqi_clean  = aqi_clean.withColumn(c, when(col(safe_c).isNull(), col(f"{alias_c}_mean")).otherwise(col(safe_c)))
    aqi_clean  = aqi_clean.drop(f"{alias_c}_mean")

# Compute AQI from PM2.5 where AQI is null
aqi_clean = aqi_clean.withColumn(
    "AQI",
    when(col("AQI").isNull(), col("`PM2_5`") * 1.5).otherwise(col("AQI"))
)

# Fill remaining AQI nulls with city-level mean
city_aqi_mean = aqi_clean.groupBy("City").agg(mean("AQI").alias("AQI_mean"))
aqi_clean     = aqi_clean.join(city_aqi_mean, on="City", how="left")
aqi_clean     = aqi_clean.withColumn("AQI", when(col("AQI").isNull(), col("AQI_mean")).otherwise(col("AQI")))
aqi_clean     = aqi_clean.drop("AQI_mean")

# Recompute AQI_Bucket from AQI value
aqi_clean = aqi_clean.withColumn("AQI_Bucket",
    when(col("AQI") <= 50,  "Good")
    .when(col("AQI") <= 100, "Satisfactory")
    .when(col("AQI") <= 200, "Moderate")
    .when(col("AQI") <= 300, "Poor")
    .when(col("AQI") <= 400, "Very Poor")
    .otherwise("Severe")
)

# Add time features
aqi_clean = aqi_clean.withColumn("Year",      year("Date"))
aqi_clean = aqi_clean.withColumn("Month",     month("Date"))
aqi_clean = aqi_clean.withColumn("DayOfWeek", dayofweek("Date"))

# Standardise city names
aqi_clean = aqi_clean.withColumn("City", trim(col("City")))

print(f"AQI clean shape: ({aqi_clean.count()}, {len(aqi_clean.columns)})")
display(aqi_clean.limit(5))


# ════════════════════════════════════════════════════════
# 2. CLEAN HEALTH DATA
# ════════════════════════════════════════════════════════

# Keep only active facilities
health_clean = health_df.filter(col("ActiveFlag_C") == "Y")

# Standardise column values
health_clean = health_clean.withColumn("State Name",    trim(col("State Name")))
health_clean = health_clean.withColumn("District Name", trim(col("District Name")))

# Aggregate to district level
health_agg = health_clean.groupBy("State Name", "District Name").agg(
    count("Facility Name").alias("total_facilities"),
    count(when(col("Facility Type") == "Hospital", True)).alias("total_hospitals")
)

print(f"Health aggregated shape: ({health_agg.count()}, {len(health_agg.columns)})")
display(health_agg.limit(5))


# ════════════════════════════════════════════════════════
# 3. CLEAN CENSUS DATA
# ════════════════════════════════════════════════════════

# Convert to pandas — easier to fix broken header
census_pd = census_df.toPandas()

# Find the row containing actual column headers
header_row_idx = None
for i, row in census_pd.iterrows():
    if any(str(v).strip().lower() in ["state", "district", "population", "area"]
           for v in row.values if v is not None):
        header_row_idx = i
        break

print(f"Header found at row index: {header_row_idx}")

# Promote that row to column names
census_pd.columns = census_pd.iloc[header_row_idx].values
census_pd         = census_pd.iloc[header_row_idx + 1:].reset_index(drop=True)

# Drop empty rows and columns
census_pd.dropna(how="all", inplace=True)
census_pd.dropna(axis=1, how="all", inplace=True)

print(f"Census columns: {list(census_pd.columns)}")
print(f"Census shape  : {census_pd.shape}")
display(spark.createDataFrame(census_pd).limit(5))

In [0]:
# ════════════════════════════════════════════════════════
# 1. FIX HEALTH DATA — check actual facility type values
# ════════════════════════════════════════════════════════

# Check what facility types actually exist
display(health_df.groupBy("Facility Type").count().orderBy("count", ascending=False))

In [0]:
# ════════════════════════════════════════════════════════
# 1. FIX HEALTH AGGREGATION
# ════════════════════════════════════════════════════════

health_agg = health_df \
    .filter(col("ActiveFlag_C") == "Y") \
    .withColumn("State Name",    trim(col("State Name"))) \
    .withColumn("District Name", trim(col("District Name"))) \
    .groupBy("District Name") \
    .agg(
        count("Facility Name").alias("total_facilities"),
        count(when(col("Facility Type").isin("dis_h", "s_t_h"), True)).alias("total_hospitals"),
        count(when(col("Facility Type") == "phc",  True)).alias("total_phc"),
        count(when(col("Facility Type") == "chc",  True)).alias("total_chc")
    )

print(f"Health agg shape: ({health_agg.count()}, {len(health_agg.columns)})")
display(health_agg.limit(5))


# ════════════════════════════════════════════════════════
# 2. GET CITY-LEVEL AQI SUMMARY
# ════════════════════════════════════════════════════════

# Aggregate AQI data to one row per city
aqi_city = aqi_clean.groupBy("City").agg(
    F.mean("AQI").alias("avg_aqi"),
    F.max("AQI").alias("max_aqi"),
    F.mean("PM2_5").alias("avg_pm25"),
    F.mean("NO2").alias("avg_no2"),
    F.mean("SO2").alias("avg_so2"),
    F.mean("CO").alias("avg_co"),
    F.min("Date").alias("data_from"),
    F.max("Date").alias("data_to")
)

print(f"AQI city summary shape: ({aqi_city.count()}, {len(aqi_city.columns)})")
display(aqi_city)


# ════════════════════════════════════════════════════════
# 3. JOIN AQI + HEALTH
# ════════════════════════════════════════════════════════

# Health is at district level, AQI is at city level
# City name ≈ District name for major cities — join on that
master_df = aqi_city.join(
    health_agg,
    aqi_city["City"] == health_agg["District Name"],
    how="left"
).drop("District Name")

print(f"Master shape: ({master_df.count()}, {len(master_df.columns)})")
display(master_df)


# ════════════════════════════════════════════════════════
# 4. CHECK UNMATCHED CITIES
# ════════════════════════════════════════════════════════

# Cities where health join failed
unmatched = master_df.filter(col("total_facilities").isNull()).select("City")
print("Cities with no health match:")
display(unmatched)

In [0]:
from pyspark.sql.functions import col, trim

# ════════════════════════════════════════════════════════
# 1. MANUAL NAME MAPPING — AQI name → Health district name
# ════════════════════════════════════════════════════════

city_mapping = {
    "Bengaluru"     : "Bangalore Urban",
    "Mumbai"        : "Mumbai Suburban",
    "Delhi"         : "North West Delhi",
    "Gurugram"      : "Gurgaon",
    "Visakhapatnam" : "Visakhapatnam",
    "Kochi"         : "Ernakulam",
    "Guwahati"      : "Kamrup Metropolitan",
    "Kolkata"       : "Kolkata",
    "Aizawl"        : "Aizawl",
    "Shillong"      : "East Khasi Hills",
    "Amaravati"     : "Guntur",
    "Brajrajnagar"  : "Jharsuguda",
    "Jorapokhar"    : "Dhanbad",
    "Talcher"       : "Angul"
}

# Convert mapping to Spark DataFrame
mapping_df = spark.createDataFrame(
    [(k, v) for k, v in city_mapping.items()],
    ["City", "Mapped_District"]
)

# ════════════════════════════════════════════════════════
# 2. REJOIN WITH CORRECTED NAMES
# ════════════════════════════════════════════════════════

# Add mapped district name to aqi_city
aqi_city_mapped = aqi_city.join(mapping_df, on="City", how="left")

# Use mapped name where available, otherwise use City directly
aqi_city_mapped = aqi_city_mapped.withColumn(
    "Join_Key",
    when(col("Mapped_District").isNotNull(), col("Mapped_District")).otherwise(col("City"))
)

# Rejoin with health data using corrected join key
master_df = aqi_city_mapped.join(
    health_agg,
    aqi_city_mapped["Join_Key"] == health_agg["District Name"],
    how="left"
).drop("District Name", "Mapped_District", "Join_Key")

print(f"Master shape: ({master_df.count()}, {len(master_df.columns)})")
display(master_df)

# ════════════════════════════════════════════════════════
# 3. VERIFY — check if unmatched cities are resolved
# ════════════════════════════════════════════════════════

still_unmatched = master_df.filter(col("total_facilities").isNull()).select("City")
print("Still unmatched after fix:")
display(still_unmatched)

In [0]:
# Fill unmatched cities with national average
from pyspark.sql.functions import avg

avg_facilities = master_df.agg(avg("total_facilities")).collect()[0][0]
avg_hospitals  = master_df.agg(avg("total_hospitals")).collect()[0][0]
avg_phc        = master_df.agg(avg("total_phc")).collect()[0][0]
avg_chc        = master_df.agg(avg("total_chc")).collect()[0][0]

master_df = master_df \
    .fillna(avg_facilities, subset=["total_facilities"]) \
    .fillna(avg_hospitals,  subset=["total_hospitals"]) \
    .fillna(avg_phc,        subset=["total_phc"]) \
    .fillna(avg_chc,        subset=["total_chc"])

print(f"Final master shape: ({master_df.count()}, {len(master_df.columns)})")
display(master_df)

In [0]:
# ════════════════════════════════════════════════════════
# 1. CONVERT TO PANDAS FOR NORMALISATION
# ════════════════════════════════════════════════════════

master_pd = master_df.toPandas()

# ════════════════════════════════════════════════════════
# 2. MIN-MAX NORMALISATION
# ════════════════════════════════════════════════════════

def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-8)

# Pollution Impact Index — higher AQI/PM2.5 = worse
master_pd["aqi_norm"]  = minmax(master_pd["avg_aqi"])
master_pd["pm25_norm"] = minmax(master_pd["avg_pm25"])

# Health Infrastructure Index — higher = better
master_pd["facilities_norm"] = minmax(master_pd["total_facilities"])
master_pd["hospitals_norm"]  = minmax(master_pd["total_hospitals"])
master_pd["phc_norm"]        = minmax(master_pd["total_phc"])
master_pd["chc_norm"]        = minmax(master_pd["total_chc"])

# ════════════════════════════════════════════════════════
# 3. COMPUTE SUB-INDICES
# ════════════════════════════════════════════════════════

# Pollution Impact Index (higher = more polluted)
master_pd["pollution_impact_index"] = (
    0.6 * master_pd["aqi_norm"] +
    0.4 * master_pd["pm25_norm"]
)

# Health Infrastructure Index (higher = better infra)
master_pd["health_infra_index"] = (
    0.4 * master_pd["hospitals_norm"] +
    0.3 * master_pd["facilities_norm"] +
    0.2 * master_pd["phc_norm"] +
    0.1 * master_pd["chc_norm"]
)

# ════════════════════════════════════════════════════════
# 4. PREPAREDNESS SCORE
# ════════════════════════════════════════════════════════

epsilon = 0.01  # avoid division by zero

# Replace the preparedness score calculation with this
master_pd["preparedness_raw"] = (
    master_pd["health_infra_index"] - 
    master_pd["pollution_impact_index"]
)

# Normalise to 0-1
master_pd["preparedness_score"] = minmax(master_pd["preparedness_raw"])

# ════════════════════════════════════════════════════════
# 5. ASSIGN TIER
# ════════════════════════════════════════════════════════

def assign_tier(score):
    if score >= 0.65:
        return "Well Prepared"
    elif score >= 0.35:
        return "Moderate Risk"
    else:
        return "Critically Vulnerable"

master_pd["tier"] = master_pd["preparedness_score"].apply(assign_tier)

# ════════════════════════════════════════════════════════
# 6. FINAL RANKING TABLE
# ════════════════════════════════════════════════════════

ranking_df = master_pd[[
    "City", "avg_aqi", "avg_pm25",
    "health_infra_index", "pollution_impact_index",
    "preparedness_score", "tier"
]].sort_values("preparedness_score", ascending=True).reset_index(drop=True)

ranking_df.insert(0, "Rank", ranking_df.index + 1)

display(spark.createDataFrame(ranking_df))

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
# import mlflow
# import mlflow.xgboost
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_absolute_error, mean_squared_error
# import numpy as np

# ════════════════════════════════════════════════════════
# 1. PREPARE TIME SERIES DATA
# ════════════════════════════════════════════════════════

# Convert AQI clean to pandas, sort by city and date
aqi_pd = aqi_clean.select("City", "Date", "AQI", "PM2_5", "Month", "DayOfWeek") \
                  .orderBy("City", "Date") \
                  .toPandas()

aqi_pd["Date"] = pd.to_datetime(aqi_pd["Date"])
aqi_pd = aqi_pd.sort_values(["City", "Date"]).reset_index(drop=True)

# ════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING — LAG + ROLLING FEATURES
# ════════════════════════════════════════════════════════

def create_features(df):
    df = df.copy()
    # Lag features
    df["aqi_lag_1"]  = df.groupby("City")["AQI"].shift(1)
    df["aqi_lag_7"]  = df.groupby("City")["AQI"].shift(7)
    df["aqi_lag_14"] = df.groupby("City")["AQI"].shift(14)
    # Rolling features
    df["aqi_roll_7_mean"] = df.groupby("City")["AQI"].transform(lambda x: x.shift(1).rolling(7).mean())
    df["aqi_roll_7_std"]  = df.groupby("City")["AQI"].transform(lambda x: x.shift(1).rolling(7).std())
    # Diwali flag — approximate dates
    diwali_dates = pd.to_datetime([
        "2015-11-11","2016-10-30","2017-10-19",
        "2018-11-07","2019-10-27","2020-11-14",
        "2021-11-04","2022-10-24"
    ])
    df["diwali_flag"] = df["Date"].apply(
        lambda d: 1 if any(abs((d - dw).days) <= 3 for dw in diwali_dates) else 0
    )
    # City one-hot encoding
    df = pd.get_dummies(df, columns=["City"], drop_first=True)
    return df

aqi_features = create_features(aqi_pd)
aqi_features.dropna(inplace=True)

# ════════════════════════════════════════════════════════
# 3. TRAIN / TEST SPLIT — last 30 days = test
# ════════════════════════════════════════════════════════

split_date = aqi_features["Date"].max() - pd.Timedelta(days=30)

train = aqi_features[aqi_features["Date"] <= split_date]
test  = aqi_features[aqi_features["Date"] >  split_date]

feature_cols = [c for c in aqi_features.columns if c not in ["Date", "AQI", "PM2_5"]]

X_train, y_train = train[feature_cols], train["AQI"]
X_test,  y_test  = test[feature_cols],  test["AQI"]

print(f"Train size : {len(X_train)}")
print(f"Test size  : {len(X_test)}")
print(f"Features   : {len(feature_cols)}")

# ════════════════════════════════════════════════════════
# 4. TRAIN XGBOOST WITH MLFLOW TRACKING
# ════════════════════════════════════════════════════════

mlflow.xgboost.autolog()

with mlflow.start_run(run_name="AirGuard_AQI_Forecast"):

    model = XGBRegressor(
        n_estimators  = 200,
        max_depth      = 5,
        learning_rate  = 0.05,
        subsample      = 0.8,
        random_state   = 42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

    mlflow.log_metric("MAE",  mae)
    mlflow.log_metric("RMSE", rmse)

    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")

# ════════════════════════════════════════════════════════
# 5. GENERATE 3-DAY FORECAST PER CITY
# ════════════════════════════════════════════════════════

def get_risk_category(aqi):
    if aqi <= 50:   return "Good"
    elif aqi <= 100: return "Satisfactory"
    elif aqi <= 200: return "Moderate"
    elif aqi <= 300: return "Poor"
    elif aqi <= 400: return "Very Poor"
    else:            return "Severe"

forecast_rows = []
cities = aqi_pd["City"].unique()

for city in cities:
    city_df = aqi_features[aqi_pd["City"] == city].sort_values("Date")
    if len(city_df) == 0:
        continue
    last_row = city_df.iloc[-1][feature_cols].values.reshape(1, -1)
    for day in range(1, 4):
        pred_aqi = model.predict(last_row)[0]
        forecast_rows.append({
            "City"        : city,
            "Day"         : f"Day +{day}",
            "Forecast_AQI": round(pred_aqi, 2),
            "Risk"        : get_risk_category(pred_aqi)
        })

forecast_df = pd.DataFrame(forecast_rows)
display(spark.createDataFrame(forecast_df))

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ════════════════════════════════════════════════════════
# 1. CITY PREPAREDNESS RANKING BAR CHART
# ════════════════════════════════════════════════════════

tier_colors = {
    "Well Prepared"       : "#2ecc71",
    "Moderate Risk"       : "#f39c12",
    "Critically Vulnerable": "#e74c3c"
}

fig1 = px.bar(
    ranking_df.sort_values("preparedness_score", ascending=True),
    x            = "preparedness_score",
    y            = "City",
    color        = "tier",
    color_discrete_map = tier_colors,
    orientation  = "h",
    title        = "City Preparedness Ranking",
    labels       = {"preparedness_score": "Preparedness Score", "City": "City"},
    hover_data   = ["avg_aqi", "health_infra_index"]
)
fig1.add_vline(x=0.35, line_dash="dash", line_color="gray")
fig1.add_vline(x=0.65, line_dash="dash", line_color="gray")
display(fig1)

# ════════════════════════════════════════════════════════
# 2. POLLUTION vs HEALTH INFRA SCATTER (BUBBLE CHART)
# ════════════════════════════════════════════════════════

fig2 = px.scatter(
    ranking_df,
    x            = "pollution_impact_index",
    y            = "health_infra_index",
    color        = "tier",
    color_discrete_map = tier_colors,
    text         = "City",
    title        = "Health Infrastructure vs Pollution Burden",
    labels       = {
        "pollution_impact_index": "Pollution Impact Index (higher = worse)",
        "health_infra_index"    : "Health Infrastructure Index (higher = better)"
    },
    size         = "avg_aqi",
    size_max     = 30
)
fig2.update_traces(textposition="top center")
fig2.add_hline(y=0.5, line_dash="dash", line_color="gray", opacity=0.5)
fig2.add_vline(x=0.5, line_dash="dash", line_color="gray", opacity=0.5)
display(fig2)

# ════════════════════════════════════════════════════════
# 3. AQI TREND + FORECAST LINE CHART (per city)
# ════════════════════════════════════════════════════════

selected_city = "Delhi"   # change to any city

# Historical AQI — last 90 days
city_history = aqi_pd[aqi_pd["City"] == selected_city].sort_values("Date").tail(90)

# Forecast for selected city
city_forecast = forecast_df[forecast_df["City"] == selected_city]
last_date     = pd.to_datetime(city_history["Date"].max())
city_forecast["Date"] = [last_date + pd.Timedelta(days=i+1) for i in range(len(city_forecast))]

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x    = city_history["Date"],
    y    = city_history["AQI"],
    mode = "lines",
    name = "Historical AQI",
    line = dict(color="#3498db", width=1.5)
))

fig3.add_trace(go.Scatter(
    x    = city_forecast["Date"],
    y    = city_forecast["Forecast_AQI"],
    mode = "lines+markers",
    name = "Forecast AQI",
    line = dict(color="#e74c3c", width=2, dash="dash")
))

fig3.update_layout(
    title  = f"AQI Trend + 3-Day Forecast — {selected_city}",
    xaxis_title = "Date",
    yaxis_title = "AQI"
)
display(fig3)

# ════════════════════════════════════════════════════════
# 4. 3-DAY FORECAST RISK TABLE (all cities)
# ════════════════════════════════════════════════════════

forecast_pivot = forecast_df.pivot(index="City", columns="Day", values=["Forecast_AQI", "Risk"])
forecast_pivot.columns = ["AQI Day+1", "AQI Day+2", "AQI Day+3", "Risk Day+1", "Risk Day+2", "Risk Day+3"]
forecast_pivot = forecast_pivot.reset_index()

display(spark.createDataFrame(forecast_pivot))

In [0]:
# ════════════════════════════════════════════════════════
# 1. PULL KEY FINDINGS FROM DATA
# ════════════════════════════════════════════════════════

most_vulnerable   = ranking_df.iloc[0]
most_prepared     = ranking_df.iloc[-1]
highest_aqi_city  = ranking_df.loc[ranking_df["avg_aqi"].idxmax()]
worst_mismatch    = ranking_df.loc[
    (ranking_df["pollution_impact_index"] - ranking_df["health_infra_index"]).idxmax()
]
best_case         = ranking_df.loc[
    (ranking_df["health_infra_index"] - ranking_df["pollution_impact_index"]).idxmax()
]

# Forecast: city with highest Day+1 AQI
worst_forecast = forecast_df[forecast_df["Day"] == "Day +1"] \
                    .sort_values("Forecast_AQI", ascending=False).iloc[0]

# ════════════════════════════════════════════════════════
# 2. DISPLAY AS HTML INSIGHT CARDS
# ════════════════════════════════════════════════════════

html = f"""
<style>
  .card-container {{ display: flex; flex-wrap: wrap; gap: 16px; padding: 16px; }}
  .card {{
    border-radius: 10px; padding: 16px; width: 280px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.1); font-family: Arial, sans-serif;
  }}
  .card h4 {{ margin: 0 0 8px 0; font-size: 13px; text-transform: uppercase; letter-spacing: 0.5px; }}
  .card p  {{ margin: 0; font-size: 15px; font-weight: bold; }}
  .card span {{ font-size: 12px; font-weight: normal; color: #555; }}
  .red    {{ background: #fdecea; border-left: 5px solid #e74c3c; }}
  .green  {{ background: #eafaf1; border-left: 5px solid #2ecc71; }}
  .orange {{ background: #fef9e7; border-left: 5px solid #f39c12; }}
  .blue   {{ background: #eaf4fb; border-left: 5px solid #3498db; }}
</style>

<div class="card-container">

  <div class="card red">
    <h4>Most Vulnerable City</h4>
    <p>{most_vulnerable['City']}</p>
    <span>Preparedness Score: {most_vulnerable['preparedness_score']:.3f} | AQI: {most_vulnerable['avg_aqi']:.1f}</span>
  </div>

  <div class="card green">
    <h4>Most Prepared City</h4>
    <p>{most_prepared['City']}</p>
    <span>Preparedness Score: {most_prepared['preparedness_score']:.3f} | AQI: {most_prepared['avg_aqi']:.1f}</span>
  </div>

  <div class="card orange">
    <h4>Highest Pollution City</h4>
    <p>{highest_aqi_city['City']}</p>
    <span>Avg AQI: {highest_aqi_city['avg_aqi']:.1f} | Score: {highest_aqi_city['preparedness_score']:.3f}</span>
  </div>

  <div class="card red">
    <h4>Worst Infra-Pollution Mismatch</h4>
    <p>{worst_mismatch['City']}</p>
    <span>High pollution, low health infra — highest risk city</span>
  </div>

  <div class="card green">
    <h4>Best Case City</h4>
    <p>{best_case['City']}</p>
    <span>Strong infra relative to pollution burden</span>
  </div>

  <div class="card blue">
    <h4>Highest Forecast AQI Tomorrow</h4>
    <p>{worst_forecast['City']}</p>
    <span>Predicted AQI: {worst_forecast['Forecast_AQI']:.1f} — {worst_forecast['Risk']}</span>
  </div>

</div>
"""

displayHTML(html)

In [0]:
# ════════════════════════════════════════════════════════
# 1. SAVE RANKING TO DELTA
# ════════════════════════════════════════════════════════

spark.createDataFrame(ranking_df) \
     .write.format("delta") \
     .mode("overwrite") \
     .saveAsTable("workspace.bronze.airguard_city_rankings")

print("Saved: airguard_city_rankings")

# ════════════════════════════════════════════════════════
# 2. SAVE FORECAST TO DELTA
# ════════════════════════════════════════════════════════

spark.createDataFrame(forecast_df) \
     .write.format("delta") \
     .mode("overwrite") \
     .saveAsTable("workspace.bronze.airguard_forecast")

print("Saved: airguard_forecast")

# ════════════════════════════════════════════════════════
# 3. MLFLOW EXPERIMENT SUMMARY
# ════════════════════════════════════════════════════════

runs = mlflow.search_runs(order_by=["start_time DESC"])

# Replace the display line with this
display(runs[[
    "run_id",
    "start_time",
    "end_time",
    "status",
    "metrics.MAE",
    "metrics.RMSE",
    "params.max_depth",
    "params.learning_rate",
    "params.num_boost_round",
    "params.subsample"
]].head(5))

print("\nNotebook complete. All outputs saved.")